# Full Scale Model Pre-Flight Check & Post-Mortem
A Jupyter notebook tailored for an HPC Environment. This notebook should be used for the "Pre-Flight Check" (testing the HeteroStruct2Seq dimension shapes and single forward passes) and the "Post-Mortem" (analyzing training loss graphs and evaluation metrics that your cluster output generated).

**WARNING: Do not execute large-scale data downloads or 150k-graph training loops directly from this notebook on an HPC login node! Use `submit_training.sh` via SLURM instead.**

## 1. Setup Environment and Hardware Check
Import required libraries, define folder structures, and verify CUDA availability to ensure the GPU is ready for the training phase.

In [ ]:
import os
import subprocess
import sys
import torch

# Ensure we operate from the project root and don't recursively jump directories
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

project_root = os.getcwd()
if project_root not in sys.path:
    sys.path.append(project_root)
    
print(f"Working Directory: {os.getcwd()}")

# Hardware check
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Target Compute Device: {device}")

## 2. Pre-Flight Check: Single Model Test
Ensure `HeteroStruct2Seq` compiles without tensor dimension errors when executing a single forward pass.

In [ ]:
from utils.model_utils import HeteroStruct2Seq

# Instantiate the newly decoupled graph model for a pre-flight sanity test
model = HeteroStruct2Seq(
    node_dim=128, 
    edge_dim=64, 
    num_layers=4, 
    vocab_size=21
)
model = model.to(device)
print("Model initialized and moved to:", device)

# Add mock forward pass data here later to verify `HeteroStruct2Seq` tensor dimensions
# dummy_output = model(...)
# print("Successful forward pass, shape:", dummy_output.shape)

## 3. Post-Mortem: Visualizing Training Results
After running `sbatch scripts/submit_training.sh` safely in the background on the HPC compute nodes, load the training history to visualize the loss curves.

In [ ]:
import json
import matplotlib.pyplot as plt

history_file = os.path.join(project_root, "outputs", "training_history.json")

if os.path.exists(history_file):
    with open(history_file, "r") as f:
        history = json.load(f)
    
    epochs = history.get("epochs", [])
    losses = history.get("losses", [])
    
    plt.figure(figsize=(10, 5))
    plt.plot(epochs, losses, label="Training Loss")
    plt.xlabel("Epochs")
    plt.ylabel("Loss")
    plt.title("HeteroStruct2Seq Training Progress")
    plt.legend()
    plt.grid(True)
    plt.show()
else:
    print(f"Training history not found at {history_file}. Run training jobs via SLURM first.")